# vxdb as agent working memory: Memory in the Loop

An agent loop (observe, reason, act) usually treats memory as something *outside* the
loop: a store queried once per turn, because a networked vector DB costs tens of
milliseconds per call. vxdb runs **in-process** and answers in ~100 microseconds, so an
agent can read and write memory on **every step**.

Four acts:

1. **The primitive in ten lines**: allocate, write, recall, destroy.
2. **With and without memory**: the same bounded-window agent fails without it and
   succeeds with it. Deterministic, offline.
3. **The agent decides what to retain**: `remember`/`recall` tools plus a system
   prompt with retention criteria; the model chooses which facts earn storage, and
   the scratchpad rejects near-duplicates *(skips itself unless `OPENAI_API_KEY`
   is set)*.
4. **The latency story**: live per-op timings from the store itself.

Requirements: `pip install vxdb`. Optional: `model2vec` for better offline embeddings,
`openai` + `openai-agents` for act 3. Every act except 3 runs fully offline.

In [ ]:
import hashlib
import math
import os
import re


def _hashed_embedder(dim=256):
    """Dependency-free fallback: hashed unigrams+bigrams, L2-normalized."""
    token_re = re.compile(r"[a-z0-9]+")

    def _vec(text):
        toks = token_re.findall(text.lower())
        grams = toks + [f"{a}_{b}" for a, b in zip(toks, toks[1:])]
        v = [0.0] * dim
        for g in grams:
            h = int.from_bytes(hashlib.md5(g.encode()).digest()[:4], "little")
            v[h % dim] += 1.0
        norm = math.sqrt(sum(x * x for x in v)) or 1.0
        return [x / norm for x in v]

    return lambda texts: [_vec(t) for t in texts]


def load_embedder():
    """OpenAI if a key is set, else model2vec, else hashed bag-of-words."""
    if os.environ.get("OPENAI_API_KEY"):
        try:
            from openai import OpenAI

            client = OpenAI()

            def embed(texts):
                resp = client.embeddings.create(
                    model="text-embedding-3-small", input=list(texts)
                )
                return [d.embedding for d in resp.data]

            return embed, "openai:text-embedding-3-small"
        except Exception as exc:
            print(f"openai unavailable ({exc!r}); falling back to a local embedder")
    try:
        from model2vec import StaticModel

        model = StaticModel.from_pretrained("minishlab/potion-retrieval-32M")
        return (lambda texts: model.encode(texts).tolist()), "model2vec potion-retrieval-32M"
    except Exception:
        return _hashed_embedder(), "hashed bag-of-words (dependency-free)"


embed, embedder_name = load_embedder()
print("embedder:", embedder_name)

## Act 1: the primitive in ten lines

`scratch()` allocates a fresh in-memory store the way you allocate a list. No server,
no connection string, no schema. Drop it when the run ends.

In [ ]:
from vxdb.agent import scratch

wm = scratch(embed)  # allocate

wm.add("the user's budget is $3000", metadata={"turn": 1})
wm.add("the user is vegetarian", metadata={"turn": 2})
wm.add("the trip is 10 days in Japan", metadata={"turn": 3})

for hit in wm.recall("how much can the user spend?", k=3):
    print(f"{hit.similarity:5.2f}  {hit.text}")

print("already noted?", wm.seen("the user budget is $3000", threshold=0.8))
wm.close()  # instant destroy

## Act 2: with and without memory

A deterministic incident-tracing task, four hops deep. Each search needs the entity
found by the previous one, and the final answer needs **all four** facts. The agent's
context window only holds the **last two** observations, like a real agent whose early
turns scrolled out of context.

- **baseline**: answers from the window alone, so early facts are gone.
- **memory**: writes each finding to a `WorkingMemory`; at answer time it recalls the
  top two notes per question and unions them with the window.

The recall *results* are real: the memory condition succeeds only if vxdb recall
actually returns the right notes for each question. The recall *invocation* is
scaffold-driven: the harness probes memory with each step's query, the model never
decides. That is one of two trigger patterns; the next section names them.

In [ ]:
import re

# (query the agent runs, document the search returns, regex extracting the entity)
HOPS = [
    (
        "what service does checkout depend on for tax calculation",
        "The checkout service depends on tax-service for tax calculation during the order flow.",
        r"depends on ([\w-]+) for tax",
    ),
    (
        "which team owns tax-service",
        "The tax-service is owned by team Vat-Wizards, who maintain its API and on-call rotation.",
        r"owned by team ([\w-]+)",
    ),
    (
        "who is the on-call engineer for team Vat-Wizards",
        "The current on-call engineer for team Vat-Wizards is Dana Ruiz.",
        r"engineer for team [\w-]+ is ([A-Za-z]+ [A-Za-z]+)",
    ),
    (
        "what is the pager id for Dana Ruiz",
        "Dana Ruiz can be paged at pager PD-4417 for production incidents.",
        r"pager (PD-\d+)",
    ),
]
REQUIRED = {"tax-service", "Vat-Wizards", "Dana Ruiz", "PD-4417"}


def extract_facts(docs):
    """Pull whichever of the four entities appear in the given documents."""
    found = set()
    for _, _, pattern in HOPS:
        for d in docs:
            m = re.search(pattern, d)
            if m:
                found.add(m.group(1))
                break
    return found

In [ ]:
WINDOW = 2  # observations still visible at answer time


def run(condition):
    wm = scratch(embed) if condition == "memory" else None
    window = []
    for _query, doc, _pattern in HOPS:  # walk the hop chain
        window = (window + [doc])[-WINDOW:]
        if wm is not None:
            wm.add(doc)  # note each finding as it happens
    if wm is not None:  # answer time: pull the facts back out of memory
        docs = [hit.text for query, _, _ in HOPS for hit in wm.recall(query, k=2)]
        docs += window  # the visible window is still available too
        wm.close()
    else:
        docs = window  # baseline: only the window survives
    return extract_facts(docs)


for condition in ("baseline", "memory"):
    facts = run(condition)
    verdict = "PASS" if facts == REQUIRED else "FAIL"
    print(f"{condition:8s} {verdict}  recovered {len(facts)}/4: {sorted(facts)}")

## Who calls recall? (the trigger question)

An agent never "just knows" to check memory. Something has to fire the recall, and the
query has to come from somewhere. Two patterns cover practice, plus a hybrid:

1. **Scaffold-driven, pre-action check** (Act 2): the harness queries memory before each
   step, and the query is the **current step context** ("about to call the billing API"),
   not a quiz question. Deterministic, costs one recall per step, and only viable because
   an in-process recall is ~100 microseconds. This is the pattern "memory in the loop"
   names.
2. **Model-driven tool call** (Act 3): `recall` is a tool and the system prompt defines
   the trigger. The prompt must cover *both* cases: the user explicitly asks ("list my
   constraints") **and** the agent is about to answer anything that depends on earlier
   context. Leave the second out and the model answers from a stale window instead of
   probing.
3. **Hybrid**: the scaffold auto-injects top-k for the current step every turn (pattern 1),
   and the model keeps the `recall` tool for deliberate probes (pattern 2).

One discipline makes all three work: a probe can only find what was **written**. If a
recall comes back with nothing relevant, the fact was never stored -- fix the write
policy (store facts the moment they appear), not the query.


## Act 3: the agent decides what to remember (OpenAI Agents SDK)

Two tools and one prompt turn `WorkingMemory` into agent memory, and the retention
decisions belong to the model, not your code. Three mechanisms share the work:

1. **The prompt sets the criteria**: store durable facts, preferences, and
   constraints the moment they appear, one `remember` call per fact; never store
   small talk; `recall` both when the user asks for earlier information **and**
   before answering anything that depends on earlier context (trigger pattern 2
   above). The full prompt is `MEMORY_INSTRUCTIONS` in the next cell.
2. **The model applies them**: it chooses which spans of conversation earn a
   `remember` call. Watch the `[tool]` trace below; the chit-chat turn should
   produce no store.
3. **The scratchpad guards the loop**: `remember` runs `wm.seen()` first, so when
   the user repeats a fact in different words, the near-duplicate is rejected
   instead of stored twice.

This cell needs `OPENAI_API_KEY` plus `pip install openai openai-agents`; it skips
itself cleanly otherwise.

In [ ]:
import os

HAVE_KEY = bool(os.environ.get("OPENAI_API_KEY"))
try:
    from agents import Agent, Runner, function_tool

    HAVE_SDK = True
except ImportError:
    HAVE_SDK = False

MEMORY_INSTRUCTIONS = (
    "You are a concise assistant. You have persistent memory tools. The MOMENT the "
    "user states any durable fact, preference, or constraint, call `remember` with a "
    "short statement of it, one call per distinct fact, including numbers like "
    "budgets and durations. Never store small talk, pleasantries, or transient "
    "chatter. Call `recall` in two situations: when the user asks you to recall or "
    "list earlier information, and BEFORE answering anything that depends on "
    "earlier context -- query it with a short description of what you need, not "
    "a quiz question. Reproduce EVERY item recall returns. Do not summarize, "
    "merge, or drop any, especially numeric ones. If recall returns nothing "
    "relevant, say you do not know rather than guessing. Your visible chat "
    "history is short, so rely on memory rather than scrollback."
)

if HAVE_KEY and HAVE_SDK:
    wm = scratch(embed)

    @function_tool
    def remember(fact: str) -> str:
        """Store one durable fact, preference, or constraint."""
        if wm.seen(fact, threshold=0.9):  # loop guard: reject near-duplicates
            print(f"  [tool] remember({fact!r}) -> already known, not stored")
            return "already known"
        wm.add(fact)
        print(f"  [tool] remember({fact!r}) -> stored")
        return "stored"

    @function_tool
    def recall(query: str) -> list[str]:
        """Return the stored facts most relevant to the query."""
        hits = [hit.text for hit in wm.recall(query, k=5)]
        print(f"  [tool] recall({query!r}) -> {len(hits)} facts")
        return hits

    agent = Agent(
        name="trip-planner",
        instructions=MEMORY_INSTRUCTIONS,
        tools=[remember, recall],
        model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
    )

    for turn in [
        "My budget is $3000 and the trip is 10 days.",
        "Nice weather today, right? Anyway, I'm vegetarian.",
        "Just to repeat: my budget is three thousand dollars.",
        "Now list every constraint I've given you.",
    ]:
        print(f"user:  {turn}")
        result = await Runner.run(agent, turn)  # noqa: F704 (top-level await is fine in Jupyter)
        print(f"agent: {result.final_output}")
        print()

    print(f"memory holds {len(wm)} facts; the model made every retention decision")
    wm.close()
else:
    print("skipped: needs OPENAI_API_KEY plus `pip install openai openai-agents`")

## Act 4: the latency story

`WorkingMemory` times every operation it performs, keeping the vxdb store op separate
from the embedding call. The store answers in microseconds. The only slow part of a
memory op is the embedder, and you pay that against any vector store.

In [ ]:
wm = scratch(embed)
for i in range(200):
    wm.add(f"note {i}: fact number {i} about the incident timeline")
for _ in range(50):
    wm.recall("what do we know about the incident?", k=5)

for op, s in wm.timing_summary().items():
    print(f"{op:9s} count={s['count']:4d}  p50={s['p50_us']:10.1f} us  ({s['p50_us'] / 1000:.3f} ms)")
wm.close()

## The takeaway

Act 4's numbers come from the store you just ran: microsecond writes and recalls,
measured live and in-process. At that latency an agent can afford to consult memory
on every step of its loop. The one per-op cost that remains is the embedder, and you
pay that against any vector store.